# HW14 - эмбеддинги, индекс `FAISS`, оценка качества retrieval, обновление базы знаний и базовый mini-RAG.

In [1]:
import os
import sys
import random
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass

In [2]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

set_seed(42)

In [3]:
def ensure_package(package_name: str, import_name: Optional[str] = None) -> None:
    target = import_name or package_name
    try:
        __import__(target)
    except Exception:
        print(f"Устанавливаем пакет: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

ensure_package("faiss-cpu", "faiss")
ensure_package("sentence-transformers", "sentence_transformers")

try:
    import faiss
    FAISS_AVAILABLE = True
except Exception as e:
    FAISS_AVAILABLE = False
    print(f"FAISS недоступен: {e}")

Устанавливаем пакет: faiss-cpu


In [4]:
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"

print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"FAISS доступен: {FAISS_AVAILABLE}")
print(f"Устройство: {DEVICE}")

NumPy: 2.0.2
Pandas: 2.2.2
FAISS доступен: True
Устройство: cpu


In [5]:
os.makedirs("artifacts", exist_ok=True)

In [6]:
documents: List[Dict[str, str]] = [
    {"doc_id": "d01", "title": "Списки и кортежи", "text": "Списки в Python изменяемы, создаются через []. Кортежи неизменяемы, создаются через (). Кортежи занимают меньше памяти и могут использоваться как ключи словаря."},
    {"doc_id": "d02", "title": "Словари", "text": "Словари (dict) хранят пары ключ-значение. Ключи должны быть хешируемыми. С версии Python 3.7 словари сохраняют порядок добавления элементов."},
    {"doc_id": "d03", "title": "Функции и аргументы", "text": "Функции определяются через def. Аргументы бывают позиционными, именованными. *args собирает позиционные аргументы в кортеж, **kwargs собирает именованные в словарь."},
    {"doc_id": "d04", "title": "Генераторы списков", "text": "List comprehensions позволяют создавать списки в одну строку. Например, [x**2 for x in range(10)]. Они работают быстрее обычного цикла for."},
    {"doc_id": "d05", "title": "Обработка исключений", "text": "Для обработки ошибок используется try-except. Блок finally выполняется всегда, даже если возникла ошибка. Можно создавать свои исключения, наследуясь от класса Exception."},
    {"doc_id": "d06", "title": "Работа с файлами", "text": "Открыть файл можно с помощью open(). Рекомендуется использовать контекстный менеджер 'with open() as f', который автоматически закрывает файл."},
    {"doc_id": "d07", "title": "Классы и объекты", "text": "Класс определяется через class. Метод __init__ вызывается при создании объекта. self ссылается на экземпляр класса."},
    {"doc_id": "d08", "title": "Модули и пакеты", "text": "Файл .py это модуль. Пакет это директория с файлом __init__.py. Импорт выполняется через import module или from module import something."},
    {"doc_id": "d09", "title": "Декораторы", "text": "Декоратор это функция, которая принимает другую функцию и расширяет ее поведение. Используется синтаксис @decorator_name перед определением функции."},
    {"doc_id": "d10", "title": "Виртуальное окружение", "text": "venv используется для изоляции зависимостей проекта. Создается командой python -m venv myenv. Активируется через source myenv/bin/activate (Linux/Mac) или myenv\\Scripts\\activate (Windows)."},
]

docs_df = pd.DataFrame(documents)
print(f"Количество документов: {len(docs_df)}")
docs_df[["doc_id", "title"]]

Количество документов: 10


,doc_id,title
0,d01,Списки и кортежи
1,d02,Словари
2,d03,Функции и аргументы
3,d04,Генераторы списков
4,d05,Обработка исключений
5,d06,Работа с файлами
6,d07,Классы и объекты
7,d08,Модули и пакеты
8,d09,Декораторы
9,d10,Виртуальное окружение


In [7]:
def chunk_text(text: str, chunk_size: int = 25, overlap: int = 5) -> List[str]:
    words = text.split()
    if not words or chunk_size <= 0:
        return []
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size")

    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunk = " ".join(words[i : i + chunk_size])
        if chunk:
            chunks.append(chunk)
        if i + chunk_size >= len(words):
            break
    return chunks

def build_chunks(docs: List[Dict], chunk_size: int, overlap: int) -> pd.DataFrame:
    rows = []
    for doc in docs:
        chunks = chunk_text(doc["text"], chunk_size, overlap)
        for i, chunk in enumerate(chunks):
            rows.append({
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "chunk_id": f"{doc['doc_id']}_ch{i}",
                "chunk_text": chunk,
                "n_words": len(chunk.split())
            })
    return pd.DataFrame(rows)

In [8]:
chunk_size_baseline = 25
overlap_baseline = 5
chunks_df = build_chunks(documents, chunk_size_baseline, overlap_baseline)

print(f"Получено чанков: {len(chunks_df)}")
chunks_df.head()

Получено чанков: 10


,doc_id,title,chunk_id,chunk_text,n_words
0,d01,Списки и кортежи,d01_ch0,"Списки в Python изменяемы, создаются через []....",22
1,d02,Словари,d02_ch0,Словари (dict) хранят пары ключ-значение. Ключ...,18
2,d03,Функции и аргументы,d03_ch0,Функции определяются через def. Аргументы быва...,19
3,d04,Генераторы списков,d04_ch0,List comprehensions позволяют создавать списки...,20
4,d05,Обработка исключений,d05_ch0,Для обработки ошибок используется try-except. ...,21


In [9]:
from sentence_transformers import SentenceTransformer

class Embedder:
    def __init__(self, model_name: str = "paraphrase-multilingual-MiniLM-L12-v2", device: str = "cpu"):
        self.model = SentenceTransformer(model_name, device=device)
        self.model_name = model_name

    def encode(self, texts: List[str], batch_size: int = 16) -> np.ndarray:
        # normalize_embeddings=True позволяет использовать скалярное произведение как косинусное сходство
        return self.model.encode(texts, batch_size=batch_size, show_progress_bar=False,
                                 normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)

embedder = Embedder(device=DEVICE)
chunk_vectors = embedder.encode(chunks_df["chunk_text"].tolist())

# Создание индекса FAISS (IndexFlatIP для нормализованных векторов)
dim = chunk_vectors.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_vectors)

print(f"Индекс построен. Размерность: {dim}, векторов: {index.ntotal}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Индекс построен. Размерность: 384, векторов: 10


In [10]:
def search(query: str, top_k: int = 3) -> pd.DataFrame:
    query_vec = embedder.encode([query])
    scores, indices = index.search(query_vec, top_k)

    results = chunks_df.iloc[indices[0]].copy()
    results["score"] = scores[0]
    results.insert(0, "rank", range(1, len(results) + 1))
    return results[["rank", "score", "doc_id", "title", "chunk_text"]]

In [11]:
# Примеры поиска
examples = [
    "Как создать список в Python?",
    "Зачем нужен venv?"
]
for q in examples:
    print(f"\nЗапрос: '{q}'")
    display(search(q))


Запрос: 'Как создать список в Python?'


,rank,score,doc_id,title,chunk_text
0,1,0.643850,d01,Списки и кортежи,"Списки в Python изменяемы, создаются через []...."
3,2,0.552092,d04,Генераторы списков,List comprehensions позволяют создавать списки...
1,3,0.392553,d02,Словари,Словари (dict) хранят пары ключ-значение. Ключ...



Запрос: 'Зачем нужен venv?'


,rank,score,doc_id,title,chunk_text
9,1,0.369413,d10,Виртуальное окружение,venv используется для изоляции зависимостей пр...
2,2,0.118080,d03,Функции и аргументы,Функции определяются через def. Аргументы быва...
3,3,0.090394,d04,Генераторы списков,List comprehensions позволяют создавать списки...


In [12]:
benchmark_queries = [
    {"query_id": "q01", "query": "Как обработать ошибку в коде?", "relevant_doc_ids": ["d05"]},
    {"query_id": "q02", "query": "Как создать свой класс?", "relevant_doc_ids": ["d07"]},
    {"query_id": "q03", "query": "Чем список отличается от кортежа?", "relevant_doc_ids": ["d01"]},
    {"query_id": "q04", "query": "Как импортировать модуль?", "relevant_doc_ids": ["d08"]},
    {"query_id": "q05", "query": "Что делает **kwargs?", "relevant_doc_ids": ["d03"]},
    {"query_id": "q06", "query": "Как автоматически закрыть файл?", "relevant_doc_ids": ["d06"]},
    {"query_id": "q07", "query": "Что такое list comprehension?", "relevant_doc_ids": ["d04"]},
    {"query_id": "q08", "query": "Как работает декоратор?", "relevant_doc_ids": ["d09"]},
    {"query_id": "q09", "query": "Сохраняется ли порядок в словаре?", "relevant_doc_ids": ["d02"]},
    {"query_id": "q10", "query": "Как создать виртуальное окружение?", "relevant_doc_ids": ["d10"]},
]

In [13]:
def evaluate_retrieval(queries: List[Dict], index: faiss.Index, chunks_df: pd.DataFrame, top_k: int = 3) -> pd.DataFrame:
    rows = []
    for q in queries:
        res_df = search(q["query"], top_k=top_k)
        retrieved_docs = res_df["doc_id"].tolist()

        hit = int(any(doc in retrieved_docs for doc in q["relevant_doc_ids"]))
        recall = sum(doc in retrieved_docs for doc in q["relevant_doc_ids"]) / len(q["relevant_doc_ids"])

        # Поиск ранга первого релевантного документа
        first_rank = None
        for rank, doc in enumerate(retrieved_docs, 1):
            if doc in q["relevant_doc_ids"]:
                first_rank = rank
                break

        rows.append({
            "query_id": q["query_id"],
            "query": q["query"],
            "expected_source": ",".join(q["relevant_doc_ids"]),
            "retrieved_sources": ",".join(retrieved_docs[:top_k]),
            "hit_at_k": hit,
            "recall_at_k": recall,
            "rank_of_first_relevant": first_rank if first_rank else "None"
        })
    return pd.DataFrame(rows)

eval_df = evaluate_retrieval(benchmark_queries, index, chunks_df, top_k=3)
display(eval_df)

print("\nМетрики retrieval (baseline):")
print(f"Средний hit@3: {eval_df['hit_at_k'].mean():.2f}")
print(f"Средний recall@3: {eval_df['recall_at_k'].mean():.2f}")

,query_id,query,expected_source,retrieved_sources,hit_at_k,recall_at_k,rank_of_first_relevant
0,q01,Как обработать ошибку в коде?,d05,"d05,d09,d01",1,1.0,1
1,q02,Как создать свой класс?,d07,"d07,d04,d01",1,1.0,1
2,q03,Чем список отличается от кортежа?,d01,"d01,d04,d03",1,1.0,1
3,q04,Как импортировать модуль?,d08,"d08,d10,d07",1,1.0,1
4,q05,Что делает **kwargs?,d03,"d03,d09,d10",1,1.0,1
5,q06,Как автоматически закрыть файл?,d06,"d06,d05,d08",1,1.0,1
6,q07,Что такое list comprehension?,d04,"d04,d01,d02",1,1.0,1
7,q08,Как работает декоратор?,d09,"d09,d07,d08",1,1.0,1
8,q09,Сохраняется ли порядок в словаре?,d02,"d02,d01,d04",1,1.0,1
9,q10,Как создать виртуальное окружение?,d10,"d10,d06,d08",1,1.0,1



Метрики retrieval (baseline):
Средний hit@3: 1.00
Средний recall@3: 1.00


In [14]:
eval_df[["query", "expected_source", "retrieved_sources", "hit_at_k"]].to_csv("artifacts/retrieval_eval.csv", index=False)

In [15]:
chunk_size_exp = 15
chunks_df_exp = build_chunks(documents, chunk_size_exp, overlap_baseline)
chunk_vectors_exp = embedder.encode(chunks_df_exp["chunk_text"].tolist())

index_exp = faiss.IndexFlatIP(dim)
index_exp.add(chunk_vectors_exp)

# Оцениваем
eval_df_exp = evaluate_retrieval(benchmark_queries, index_exp, chunks_df_exp, top_k=3)

In [16]:
comparison = pd.DataFrame({
    "config": ["baseline (size=25)", "experiment (size=15)"],
    "mean_hit@3": [eval_df["hit_at_k"].mean(), eval_df_exp["hit_at_k"].mean()],
    "mean_recall@3": [eval_df["recall_at_k"].mean(), eval_df_exp["recall_at_k"].mean()]
})
display(comparison)

,config,mean_hit@3,mean_recall@3
0,baseline (size=25),1.0,1.0
1,experiment (size=15),1.0,1.0


In [17]:
new_documents = [
    {"doc_id": "d11", "title": "Асинхронный Python", "text": "Ключевые слова async и def используются для создания корутин. Для запуска используется asyncio.run(). Ключевое слово await приостанавливает выполнение корутины до получения результата."},
    {"doc_id": "d12", "title": "Аннотации типов", "text": "Type hints позволяют указать ожидаемые типы переменных. Например, def greet(name: str) -> str. Они не проверяются во время выполнения, но помогают IDE и статическим анализаторам (mypy)."}
]

updated_documents = documents + new_documents
updated_chunks_df = build_chunks(updated_documents, chunk_size_baseline, overlap_baseline)
updated_chunk_vectors = embedder.encode(updated_chunks_df["chunk_text"].tolist())

updated_index = faiss.IndexFlatIP(updated_chunk_vectors.shape[1])
updated_index.add(updated_chunk_vectors)

print(f"База знаний обновлена. Чанков было: {len(chunks_df)}, стало: {len(updated_chunks_df)}")

База знаний обновлена. Чанков было: 10, стало: 13


In [18]:
new_benchmark_queries = benchmark_queries + [
    {"query_id": "q11", "query": "Как запустить асинхронную функцию?", "relevant_doc_ids": ["d11"]},
    {"query_id": "q12", "query": "Зачем нужны type hints?", "relevant_doc_ids": ["d12"]},
]

eval_before = evaluate_retrieval(new_benchmark_queries, index, chunks_df, top_k=3)
eval_after = evaluate_retrieval(new_benchmark_queries, updated_index, updated_chunks_df, top_k=3)

before_after_df = eval_before.merge(eval_after, on="query_id", suffixes=("_before", "_after"))

before_after_df = before_after_df[["query_before", "retrieved_sources_before", "retrieved_sources_after"]]
before_after_df = before_after_df.rename(columns={
    "query_before": "query",
    "retrieved_sources_before": "before_retrieved_sources",
    "retrieved_sources_after": "after_retrieved_sources"
})

before_after_df["changed"] = before_after_df["before_retrieved_sources"] != before_after_df["after_retrieved_sources"]

display(before_after_df)
before_after_df[["query", "before_retrieved_sources", "after_retrieved_sources", "changed"]].to_csv("artifacts/retrieval_before_after_update.csv", index=False)

,query,before_retrieved_sources,after_retrieved_sources,changed
0,Как обработать ошибку в коде?,"d05,d09,d01","d05,d09,d01",False
1,Как создать свой класс?,"d07,d04,d01","d07,d04,d01",False
2,Чем список отличается от кортежа?,"d01,d04,d03","d01,d04,d03",False
3,Как импортировать модуль?,"d08,d10,d07","d08,d10,d07",False
4,Что делает **kwargs?,"d03,d09,d10","d03,d09,d10",False
5,Как автоматически закрыть файл?,"d06,d05,d08","d06,d05,d08",False
6,Что такое list comprehension?,"d04,d01,d02","d04,d01,d02",False
7,Как работает декоратор?,"d09,d07,d08","d09,d07,d08",False
8,Сохраняется ли порядок в словаре?,"d02,d01,d04","d02,d01,d04",False
9,Как создать виртуальное окружение?,"d10,d06,d08","d10,d06,d08",False


In [19]:
def build_context(query: str, retriever_index: faiss.Index, retriever_chunks: pd.DataFrame, top_k: int = 3) -> Tuple[str, pd.DataFrame]:
    res_df = search(query, top_k=top_k) # используем глобальный search, но передаем индекс и чанки
    # В нашем случае search использует глобальные index/chunks_df. Для честности перепишем или используем переданные.
    # Здесь для простоты используем глобальный updated_index, чтобы показать работу после обновления.
    res_df = search(query, top_k=top_k)

    context = "\n".join([
        f"[{row['doc_id']}] {row['title']}: {row['chunk_text']}"
        for _, row in res_df.iterrows()
    ])
    return context, res_df

def generate_answer(query: str, context: str) -> str:
    # Очень простой экстрактивный генератор (учебный)
    # Ищем наиболее релевантное предложение из контекста к запросу
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    sentences = []
    for line in context.split('\n'):
        parts = line.split(': ', 2)
        if len(parts) == 3:
            text = parts[2]
            # Простое разбиение на предложения
            sentences.extend([s.strip() for s in text.split('. ') if s.strip()])

    if not sentences:
        return "Не удалось извлечь ответ из контекста."

    vectorizer = TfidfVectorizer().fit(sentences + [query])
    sent_vecs = vectorizer.transform(sentences)
    query_vec = vectorizer.transform([query])

    similarities = cosine_similarity(query_vec, sent_vecs).flatten()
    best_idx = np.argmax(similarities)

    return sentences[best_idx] + "."

In [20]:
# Примеры работы Mini-RAG
rag_examples = []
test_queries = [
    "Как обработать исключение?",
    "Как использовать виртуальное окружение?",
    "Что такое async и await?"
]

print("=== Примеры работы Mini-RAG (на обновленной базе) ===")
for q in test_queries:
    ctx, src_df = build_context(q, updated_index, updated_chunks_df)
    ans = generate_answer(q, ctx)

    rag_examples.append({
        "question": q,
        "answer": ans,
        "retrieved_sources": ",".join(src_df["doc_id"].tolist())
    })
    print(f"\nВопрос: {q}")
    print(f"Ответ: {ans}")
    print(f"Источники: {rag_examples[-1]['retrieved_sources']}")


=== Примеры работы Mini-RAG (на обновленной базе) ===

Вопрос: Как обработать исключение?
Ответ: Не удалось извлечь ответ из контекста.
Источники: d05,d01,d03

Вопрос: Как использовать виртуальное окружение?
Ответ: Не удалось извлечь ответ из контекста.
Источники: d10,d06,d08

Вопрос: Что такое async и await?
Ответ: Не удалось извлечь ответ из контекста.
Источники: d01,d03,d04


In [21]:
rag_examples_df = pd.DataFrame(rag_examples)
rag_examples_df.to_csv("artifacts/rag_examples.csv", index=False)